# Implementación computacional del algoritmo greedy de partición de intervalos

En esta sección se presenta la implementación computacional del algoritmo greedy de partición de intervalos aplicado al problema de asignación de aeronaves a vuelos programados.

Cada vuelo se modela como un intervalo temporal cerrado, durante el cual una aeronave se considera completamente ocupada. El objetivo es asignar a cada vuelo una aeronave, de manera que no existan solapamientos temporales en una misma aeronave y que el número total de aeronaves utilizadas sea mínimo.

La implementación se realiza en un cuaderno de Jupyter Notebook con el propósito de favorecer la claridad, la reproducibilidad de los experimentos y la comprensión progresiva del flujo algorítmico.

## Importación de librerías

In [1]:
import random 
import heapq
import pandas as pd

## Parámetros del modelo sintético
El tiempo se discretiza en bloques operativos reales de 15 minutos, lo que permite representar horarios compatibles con la práctica operacional del transporte aéreo comercial.

In [2]:
BLOQUE_TIEMPO = 0.25   # 15 minutos = 0.25 horas
HORIZONTE = 72.0       # Horizonte temporal de planificación (horas)

## Funciones auxiliares
Estas funciones permiten discretizar el tiempo y presentar los resultados en un formato comprensible para el lector.

In [3]:
def redondear_a_bloque(valor, bloque):
    """
    Redondea un valor continuo al múltiplo más cercano del bloque.
    """
    return round(valor / bloque) * bloque


def convertir_horas_a_hhmm(horas):
    """
    Convierte horas decimales al formato HH:MM.
    """
    h = int(horas)
    m = int(round((horas - h) * 60))
    return f"{h:02d}:{m:02d}"

## Generación de instancias sintéticas de vuelos

Cada vuelo se modela como un intervalo temporal cerrado $[s_i, f_i]$, con tiempos de inicio y duración discretizados para reflejar horarios operativos reales.

In [4]:
def generar_vuelos_sinteticos(
    numero_vuelos=25,
    duracion_vuelo=(1.0, 9.0),      
    semilla=42
):
    random.seed(semilla)
    vuelos = []

    for identificador in range(1, numero_vuelos + 1):

        inicio = redondear_a_bloque(
            random.uniform(0, HORIZONTE - duracion_vuelo[1]),
            BLOQUE_TIEMPO
        )

        duracion = redondear_a_bloque(
            random.uniform(*duracion_vuelo),
            BLOQUE_TIEMPO
        )

        fin = inicio + duracion

        if fin <= HORIZONTE:
            vuelos.append((identificador, inicio, fin))

    return vuelos

In [5]:
def construir_tabla_instancia(vuelos):
    """
    Construye un DataFrame con los vuelos generados (instancia sintética).
    Incluye: Vuelo, Inicio, Fin, Duración (en HH:MM).
    """
    registros = []
    for vuelo, inicio, fin in vuelos:
        duracion = fin - inicio
        registros.append((
            vuelo,
            convertir_horas_a_hhmm(inicio),
            convertir_horas_a_hhmm(fin),
            convertir_horas_a_hhmm(duracion)
        ))
    return pd.DataFrame(
        registros,
        columns=["Vuelo", "Inicio", "Fin", "Duración"]
    )

In [6]:
vuelos = generar_vuelos_sinteticos()

print("Instancia de vuelos generados (orden de creación):")
tabla_instancia = construir_tabla_instancia(vuelos)
display(tabla_instancia)

Instancia de vuelos generados (orden de creación):


,Vuelo,Inicio,Fin,Duración
0,1,40:15,41:30,01:15
1,2,17:15,20:00,02:45
2,3,46:30,53:00,06:30
3,4,56:15,58:00,01:45
4,5,26:30,27:45,01:15
5,6,13:45,18:45,05:00
6,7,01:45,04:15,02:30
7,8,41:00,46:15,05:15
8,9,14:00,19:45,05:45
9,10,51:00,52:00,01:00


## Profundidad del conjunto de intervalos

La profundidad se define como el máximo número de vuelos que se solapan en un mismo instante y constituye una cota inferior sobre el número de aeronaves necesarias.

In [7]:
def calcular_profundidad(intervalos):
    eventos = []

    for vuelo, inicio, fin in intervalos:
        eventos.append((inicio, +1))
        eventos.append((fin, -1))  # intervalos cerrados

    eventos.sort()

    vuelos_activos = 0
    profundidad = 0

    for tiempo, cambio in eventos:
        vuelos_activos += cambio
        profundidad = max(profundidad, vuelos_activos)

    return profundidad

## Algoritmo greedy de partición de intervalos

El algoritmo procesa los vuelos en orden no decreciente de tiempo de inicio y utiliza una cola de prioridades para reutilizar aeronaves disponibles siempre que sea posible.

In [8]:
def obtener_inicio(vuelo):
    return vuelo[1]


def asignar_aeronaves(intervalos):

    vuelos_ordenados = sorted(intervalos, key=obtener_inicio)

    cola_prioridad = []   # (tiempo_fin, aeronave)
    asignacion = []
    total_aeronaves = 0

    for vuelo, inicio, fin in vuelos_ordenados:

        if cola_prioridad and cola_prioridad[0][0] < inicio:
            tiempo_fin, aeronave = heapq.heappop(cola_prioridad)
        else:
            total_aeronaves += 1
            aeronave = total_aeronaves

        heapq.heappush(cola_prioridad, (fin, aeronave))
        asignacion.append((vuelo, inicio, fin, aeronave))

    return asignacion, total_aeronaves

In [9]:
def construir_tabla(asignacion):
    """
    Construye un DataFrame con los resultados de la asignación.
    Incluye columnas: Vuelo, Inicio, Fin, Duración, Aeronave y Matrícula.
    La duración se muestra en formato HH:MM.
    """
    registros = []

    for vuelo, inicio, fin, aeronave in asignacion:
        matricula = f"XA-{aeronave:04d}"
        duracion = fin - inicio
        duracion_str = convertir_horas_a_hhmm(duracion)
        registros.append((
            vuelo,
            convertir_horas_a_hhmm(inicio),
            convertir_horas_a_hhmm(fin),
            duracion_str,
            aeronave,
            matricula
        ))

    return pd.DataFrame(
        registros,
        columns=["Vuelo", "Inicio", "Fin", "Duración", "Aeronave", "Matrícula"]
    )

## Ejecución del experimento computacional

En esta celda se ejecuta el experimento completo y se muestran los resultados obtenidos.

In [10]:
vuelos = generar_vuelos_sinteticos()

profundidad = calcular_profundidad(vuelos)

asignacion, aeronaves_usadas = asignar_aeronaves(vuelos)

tabla_resultados = construir_tabla(asignacion)

print(f"Profundidad del conjunto de intervalos: {profundidad}")
print(f"Aeronaves utilizadas por el algoritmo: {aeronaves_usadas}")

tabla_resultados

Profundidad del conjunto de intervalos: 6
Aeronaves utilizadas por el algoritmo: 6


,Vuelo,Inicio,Fin,Duración,Aeronave,Matrícula
0,7,01:45,04:15,02:30,1,XA-0001
1,23,05:00,07:45,02:45,1,XA-0001
2,14,05:45,07:30,01:45,2,XA-0002
3,24,06:15,09:30,03:15,3,XA-0003
4,6,13:45,18:45,05:00,2,XA-0002
5,9,14:00,19:45,05:45,1,XA-0001
6,22,14:15,17:30,03:15,3,XA-0003
7,2,17:15,20:00,02:45,4,XA-0004
8,12,21:30,23:45,02:15,3,XA-0003
9,18,23:45,29:15,05:30,2,XA-0002
